In [33]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [34]:
df = pd.read_csv('../data/processed/rq2_features_daylight.csv', index_col='Datetime (UTC)', parse_dates=True)

In [35]:
features= ['Gb(i)', 'Gd(i)', 'Gr(i)', 'H_sun', 'T2m', 'WS10m', 'P_lag_1h', 'P_lag_24h', 'hour', 'month']

In [36]:
X = df[features]
y = df['P']

In [37]:
train_end = int(len(df) * 0.70)
validation_end = int(len(df) * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val     = X.iloc[train_end:validation_end], y.iloc[train_end:validation_end]
X_test, y_test   = X.iloc[validation_end:], y.iloc[validation_end:]

In [38]:
model_v2 = XGBRegressor(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    min_child_weight=10,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=2,
    reg_lambda=2,
    random_state=42
)

model_v2.fit(X_train, y_train)

y_train_pred_v2 = model_v2.predict(X_train)
y_test_pred_v2 = model_v2.predict(X_test)

In [39]:
print("V2 TRAIN ->",
      f"MAE: {mean_absolute_error(y_train, y_train_pred_v2):.2f}",
      f"R²: {r2_score(y_train, y_train_pred_v2)*100:.1f}%")
print("V2 TEST  ->",
      f"MAE: {mean_absolute_error(y_test, y_test_pred_v2):.2f}",
      f"R²: {r2_score(y_test, y_test_pred_v2)*100:.1f}%")

V2 TRAIN -> MAE: 9.83 R²: 98.6%
V2 TEST  -> MAE: 60.30 R²: -146.4%


In [40]:
from sklearn.model_selection import KFold, cross_val_score

model_v3 = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42
)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = -cross_val_score(model_v3, X, y, cv=kfold, scoring='neg_mean_absolute_error')
r2_scores = cross_val_score(model_v3, X, y, cv=kfold, scoring='r2')

print(f"MAE по fold: {mae_scores}")
print(f"MAE просек: {mae_scores.mean():.2f} (+/- {mae_scores.std():.2f})")
print(f"R² по fold: {r2_scores}")
print(f"R² просек: {r2_scores.mean()*100:.1f}% (+/- {r2_scores.std()*100:.1f})")

MAE по fold: [6.66948596 6.69728052 6.51249494 5.97857876 5.74113038]
MAE просек: 6.32 (+/- 0.39)
R² по fold: [0.9911115  0.99169197 0.99179191 0.99359711 0.99390521]
R² просек: 99.2% (+/- 0.1)


In [41]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importance)

Gr(i)        0.706970
P_lag_1h     0.099361
month        0.090920
Gb(i)        0.036591
hour         0.016822
H_sun        0.016114
Gd(i)        0.015553
P_lag_24h    0.013127
T2m          0.002633
WS10m        0.001907
dtype: float32
